In [ ]:
# Copyright (c) 2026 Nokia Bell Labs
# Licensed under the BSD 3 Clause license
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import re
import pandas as pd
import os
import sys
import numpy as np
from matplotlib.colors import Normalize
import seaborn as sns
import ast
from sklearn.metrics import adjusted_rand_score
import csv

In [ ]:
def extract_ground_truth_list(run_id, base_log_dir):
    #pattern = r"Ground Truth Client to Cluster:\s*\[([0-9,\s]+)\]"
    pattern = re.compile(r"Ground Truth Client to Cluster:\s*(\[[^\]]+\])")

    log_dir = os.path.join(base_log_dir, str(run_id))
    file_name = str(run_id) + "_pfl_experiments.log"
    file_path = os.path.join(log_dir, file_name)

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            #atch = re.search(pattern, line)
           

            match = pattern.match(line)
            if match:
                raw_list_str = match.group(1)  # This will be the string inside [...]
                cluster_assignments = ast.literal_eval(raw_list_str)  # Safely turns string into list
                print("Parsed list:", cluster_assignments)
                converted_list = [int(cluster_assignments[i]) for i in range(len(cluster_assignments))]
                return converted_list

    print("Line not found or could not extract list.")
    return None

In [ ]:
def compute_ari(list_labels, dict_labels):

    if len(list_labels) != len(dict_labels):
        return None
    
    converted_dict = {int(k): v for k, v in dict_labels.items()}
    converted_list = [int(list_labels[i]) for i in range(len(list_labels))]

    # Convert dict to list using index order
    dict_label_list = [converted_dict[i] for i in range(len(converted_list))]

    # Compute ARI
    ari_score = adjusted_rand_score(converted_list, dict_label_list)
    return ari_score

In [ ]:
def post_process_algs_for_writing_to_csv(algs_list):
    all_algorithms1 = [alg for alg in algs_list if alg != 'centralized']
    all_algorithms2 = ['FedAvg' if x == 'vanillaFL' else x for x in all_algorithms1]
    alg_name_map = {x: 'vanillaFL' if x == 'FedAvg' else x for x in all_algorithms2}
    
    return all_algorithms2, alg_name_map

In [ ]:
def export_results_to_csv(results_list, loss_info):
    """
    Creates a matrix of mean ± std for each algorithm across datasets and saves it to a CSV file.

    Args:
        results_list (list): List of dicts with keys 'dataset', 'avgs', 'stds'
        loss_info (dict): keys relevant to the results_list map e.g. Output CSV file path
    """

    avg_key = loss_info['mean']
    std_key = loss_info['std']
    ds_key = loss_info['dataset']
    output_filename = loss_info['file']

    # Step 1: Get all dataset names
    dataset_names = [entry[ds_key] for entry in results_list]

    # Step 2: Get all algorithm names from the first entry's 'avgs'
    all_algorithms = sorted(results_list[0][avg_key].keys())
    all_algorithms, alg_name_map = post_process_algs_for_writing_to_csv(all_algorithms)

    # Step 3: Build matrix: rows = algs, cols = datasets
    matrix = {alg: {} for alg in all_algorithms}

    for entry in results_list:
        dataset = entry[ds_key]
        avgs = entry[avg_key]
        stds = entry[std_key]
        for alg in all_algorithms:
            mean = avgs.get(alg_name_map[alg], None)
            std = stds.get(alg_name_map[alg], None)
            if mean is not None and std is not None:
                matrix[alg][dataset] = f"{mean:.3f} ± {std:.3f}"
            else:
                matrix[alg][dataset] = "N/A"

    # Step 4: Write to CSV
    with open(output_filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Header row
        writer.writerow(['Algorithm'] + dataset_names)
        # Data rows
        for alg in all_algorithms:
            row = [alg] + [matrix[alg].get(ds, "N/A") for ds in dataset_names]
            writer.writerow(row)

    print(f"CSV file saved to {output_filename}")


In [ ]:
def compute_avg_and_std_by_alg(data):
    """
    Computes the average and standard deviation of values grouped by algorithm name.

    Args:
        data (dict): A dictionary with keys like "alg=centralized, seed=1.txt" and float values.

    Returns:
        tuple: (avg_map, std_map)
            - avg_map: dict mapping algorithm name to average value
            - std_map: dict mapping algorithm name to standard deviation
    """
    grouped = defaultdict(list)

    # Group values by algorithm
    for key, value in data.items():
        #print(f'in compute: {key}')
        parts = key.split(",")
        alg_part = parts[0].strip()  # e.g., "alg=centralized"
        alg_name = alg_part.split("=")[1]  # e.g., "centralized"
        #print(alg_name)
        grouped[alg_name].append(value)

    # Compute average and standard deviation
    avg_map = {}
    std_map = {}

    for alg, values in grouped.items():
        avg_map[alg] = float(np.mean(values))
        std_map[alg] = float(np.std(values))

    return avg_map, std_map

In [ ]:
def parse_files_in_directory(directory):
    data = defaultdict(dict)  # Dictionary to hold the parsed data for all files

    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory, filename)

            with open(filepath, "r") as file:
                file_data = {"pairs": defaultdict(list), "json_data": None}
                lines = file.readlines()

                # Process all lines except the JSON part
                json_lines = []
                for line in lines:
                    line = line.strip()
                    if line.startswith("key="):
                        # Extract key and value while accounting for "=" and "," in the content
                        try:
                            prefix, key_value_pair = line.split("key=", 1)
                            key, value = key_value_pair.split(", value=", 1)
                            # Default dict auto handles initialization to empty list
                            file_data["pairs"][key].append(value)
                        except ValueError:
                            print(f"Skipping malformed line: {line}")
                    else:
                        # Collect all JSON lines
                        json_lines.append(line)

                # Combine JSON lines and parse them
                if json_lines:
                    try:
                        combined_json = "\n".join(json_lines)
                        file_data["json_data"] = json.loads(combined_json)
                    except json.JSONDecodeError:
                        print(f"Skipping invalid JSON in file: {filename}")

                data[filename] = file_data

    return data

In [ ]:
def extract_losses(data_list, exp_list, same_ds):

    ret_val = []
    
    for data_item in data_list:
        
        print(f"On experiment {data_item['id']}")
        data = data_item['results']
        row = data_item['meta_data']
        gt = data_item['gt']

        dataset = row['Dataset']
        if same_ds:
            dataset = f"{dataset}_{data_item['id']}"

        losses_map = {}
        ari_map = {}
        # Process each file in the data
        for filename, content in data.items():
            # Access the pairs data
            loss_matrix_json = content.get('json_data', {})

            # if filename not in file_id_dict.keys():
            #     continue
            #print(filename)

            if 'loss_matrix_test' in loss_matrix_json and "client_to_selected_model_mapping_dict" in loss_matrix_json:
            # if 'loss_matrix_test' in loss_matrix_json and "selected_model_to_client_mapping_dict" in loss_matrix_json:
                loss_matrix = np.array(loss_matrix_json['loss_matrix_test'])
                last_matrix = loss_matrix[loss_matrix.shape[0] -1]
                #print(f"got loss matrix for {filename}, shape = {last_matrix.shape}")
                #print(last_matrix)

                # mapping_mat = np.array(loss_matrix_json['selected_model_to_client_mapping_dict'])
                # last_map_dict = mapping_mat[mapping_mat.shape[0] - 1]
                # print(f"got mapping matrix for {filename}")
                # print(last_map_dict)
                # rev_map = get_rev_map_dict(last_map_dict)
                # print(rev_map)
                # ordered_models_indx = [rev_map[rows_to_keep[i]] for i in range(len(rows_to_keep))]

                rev_mapping_mat = np.array(loss_matrix_json['client_to_selected_model_mapping_dict'])
                client_to_model_map = rev_mapping_mat[rev_mapping_mat.shape[0] - 1]
                #print(f"client_to_model_map = {client_to_model_map}")

                # Get losses for each user's assigned attribute
                selected_losses = [
                    last_matrix[int(cid)][model_id]
                    for cid, model_id in client_to_model_map.items()
                ]

                # Average loss
                average_loss = np.mean(selected_losses)
                losses_map[filename] = average_loss
                ari_map[filename] = compute_ari(gt, client_to_model_map)
                #print("Per-client assigned attribute losses:", selected_losses)
                #print("Average loss:", average_loss)

        avg_map, std_map = compute_avg_and_std_by_alg(losses_map)    
        avg_ari_map, std_ari_map = compute_avg_and_std_by_alg(ari_map)  
        #print(f"avgs: {avg_map}")
        #print(f"stds: {std_map}")
        ret_val.append ( {'dataset': dataset, 'mean': avg_map, 'std': std_map, 'ari_mean': avg_ari_map, 'ari_std': std_ari_map})

    return ret_val

In [ ]:
class StatClass:
    def __init__(self, base_path, results_dir='results', exp_drv_dir='experiment_driver', drvr_file_name='experiment_list.csv'):
        self.df = None
        self.base_path = base_path
        full_input_file_path = os.path.join(self.base_path, exp_drv_dir, drvr_file_name)

        if not os.path.exists(full_input_file_path):
            print("There is no driver file so exiting out")
            sys.exit(1)
        else:
            self.df = pd.read_csv(full_input_file_path)

        self.results_dir = os.path.join(self.base_path, results_dir)

    def compare_losses(self, exps_list=[], same_ds=False):
        data_list = []
        try:
            if self.df is not None:
                for index, row in self.df.iterrows():
                    exp_id_int = row['Experiment ID']
                    if exp_id_int not in exps_list:
                        continue

                    gt = extract_ground_truth_list(exp_id_int, self.results_dir)

                    experiment_id = str(exp_id_int)
                    result_path = os.path.join(self.results_dir, experiment_id)
                    result_data = parse_files_in_directory(result_path)
                    data_list.append({'id': experiment_id, 'results': result_data, 'meta_data': row, 'gt': gt})
            else:
                print("Data has not been loaded yet. Please 'read_file' first.")
                return
        except ValueError:
            print("Error parsing file.")
            return

        return extract_losses(data_list, exps_list, same_ds)

In [ ]:
directory_path = "../"
results_dir='results'
statter = StatClass(directory_path, results_dir)
base_log_dir = os.path.join(directory_path, results_dir)

exps_to_include = [101, 201, 301]

In [ ]:
results = statter.compare_losses(exps_list=exps_to_include)
print(results)

In [ ]:
out_file_path = os.path.join(directory_path, 'results', 'unsupervised_table_loss.csv')
loss_info = {'mean': 'mean', 'std': 'std', 'dataset': 'dataset', 'file': out_file_path}
export_results_to_csv(results, loss_info)
out_file_path = os.path.join(directory_path, 'results', 'unsupervised_table_ARI.csv')
loss_info = {'mean': 'ari_mean', 'std': 'ari_std', 'dataset': 'dataset', 'file': out_file_path}
export_results_to_csv(results, loss_info)

In [ ]:
for run_id in exps_to_include:
    gt = extract_ground_truth_list(run_id, base_log_dir)
    print(f'id:{run_id}: gt {gt}')

Newer experiments

In [ ]:
exps_to_include = [2101, 2102, 2103, 2104]
results = statter.compare_losses(exps_list=exps_to_include, same_ds=True)
print(results)

In [ ]:
out_file_path = os.path.join(directory_path, 'results', 'unsupervised_table_loss_scaling.csv')
loss_info = {'mean': 'mean', 'std': 'std', 'dataset': 'dataset', 'file': out_file_path}
export_results_to_csv(results, loss_info)
out_file_path = os.path.join(directory_path, 'results', 'unsupervised_table_ARI_scaling.csv')
loss_info = {'mean': 'ari_mean', 'std': 'ari_std', 'dataset': 'dataset', 'file': out_file_path}
export_results_to_csv(results, loss_info)

In [ ]:
for run_id in exps_to_include:
    gt = extract_ground_truth_list(run_id, base_log_dir)
    print(f'id:{run_id}: gt {gt}')